# Step 5: Questions, key facts and parallel versions (Mistral via Ollama)

For each candidate gold chunk (`data/gold/candidates.jsonl`, best first) Mistral writes, in the chunk's language:
a **question**, a short **answer** copied from the chunk, and a **key fact** sentence.
A second call translates all three into the other language (the **parallel version**).

Every result is checked automatically; failed checks are sent back to the model (up to 5 tries for the question,
3 for the translation). Generation stops once a language has `TARGET_PER_LANGUAGE` accepted questions.

| # | Rule | Check |
|---|---|---|
| 1 | Question stands on its own (no "the passage", "the author", "this", "he", "im Text", "dieser", "wir" …) | banned-word list |
| 2 | Short factual answer, 1–8 words, copied from the chunk | answer found in chunk |
| 3 | Answer not given away in the question | whole-word match |
| 4 | Wording close to the chunk | ≥ 70 % of content words from the chunk |
| 5 | Only this chunk answers it | BM25 ranks it first, and none of the next 5 chunks contain the answer |
| 6 | One question, ≤ 30 words, no figure/page/literature references | regex |
| 7 | Key fact: one sentence, chunk wording, contains the answer | answer in key fact, ≥ 70 % overlap |
| 8 | Parallel version in the other language | language, same numbers, parallel answer in parallel key fact (word endings may differ) and not in parallel question |

**Output:** `data/questions/questions.jsonl`, `data/questions/rejected.jsonl`, and the review sheet `review/03_questions.csv`.

**Runs.** Accepted questions are never regenerated. With `RETRY_REJECTED = True` the chunks rejected in earlier
runs are tried again with new seeds (set `RUN` to a new number each time); the earlier rejections are kept in
`data/questions/rejected_previous.jsonl`.

In [ ]:
import collections
import json
import re
import sys
import time
from pathlib import Path

import pandas as pd
import requests

# The notebook lives in notebooks/, the repository root is one level up.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "ingest"))
from lexical import BM25, tokenize  # same tokenizer/stopwords as the candidate selection

CANDIDATES = ROOT / "data" / "gold" / "candidates.jsonl"
ALL_CHUNKS = ROOT / "data" / "chunks" / "chunks.jsonl"
OUT_DIR = ROOT / "data" / "questions"
QUESTIONS_PATH = OUT_DIR / "questions.jsonl"
REJECTED_PATH = OUT_DIR / "rejected.jsonl"
PREVIOUS_REJECTED_PATH = OUT_DIR / "rejected_previous.jsonl"
REVIEW_DIR = ROOT / "review"
REVIEW_PATH = REVIEW_DIR / "03_questions.csv"

OLLAMA = "http://localhost:11434"
MODEL = "mistral"            # any model pulled with `ollama pull ...`
TEMPERATURE = 0.2            # low = closer to the chunk's wording, more repeatable
SEED = 42                    # fixed seed so a re-run gives the same results
RUN = 2                      # increase for every new run: gives new seeds for retried chunks
RETRY_REJECTED = True        # try the chunks rejected in earlier runs again
MAX_ATTEMPTS = 5             # tries for question/answer/key fact
MAX_PARALLEL_ATTEMPTS = 3    # tries for the translation
TARGET_PER_LANGUAGE = 75     # 60 needed for the experiment + spares for your review

# Thresholds for the checks
MIN_WORD_OVERLAP = 0.7
MAX_ANSWER_WORDS = 8
MAX_QUESTION_WORDS = 30
MAX_KEY_FACT_WORDS = 40
UNIQUE_TOP_N = 5             # the answer must not appear in the next 5 BM25 hits
MIN_ANSWER_MATCH = 0.7       # share of the translated answer's words that must appear in the translated key fact

LANG_NAME = {"DE": "German", "EN": "English"}
OTHER = {"DE": "EN", "EN": "DE"}
print("Repository root:", ROOT)

In [ ]:
try:
    tags = requests.get(f"{OLLAMA}/api/tags", timeout=5).json()
except requests.ConnectionError:
    raise SystemExit("Ollama is not running. Start the Ollama app or run `ollama serve`.")

installed = [m["name"] for m in tags.get("models", [])]
print("Installed models:", installed)
assert any(n.split(":")[0] == MODEL.split(":")[0] for n in installed), f"Run: ollama pull {MODEL}"

In [ ]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

candidates = read_jsonl(CANDIDATES)
all_chunks = read_jsonl(ALL_CHUNKS)
chunk_text = {c["chunk_id"]: c["text"] for c in all_chunks}

# One BM25 index per language over ALL chunks, so uniqueness is checked against everything
index = {}
for lang in LANG_NAME:
    lang_chunks = [c for c in all_chunks if c["language"] == lang]
    index[lang] = {"ids": [c["chunk_id"] for c in lang_chunks],
                   "bm25": BM25([tokenize(c["text"], lang) for c in lang_chunks])}

# Results from the very first run have a different format and would be skipped by mistake
for path in (QUESTIONS_PATH, REJECTED_PATH):
    if path.exists():
        old = read_jsonl(path)
        if old and "origin_lang" not in old[0]:
            raise SystemExit(f"{path.name} is from the first run. Delete it first: "
                             f"git rm {path.relative_to(ROOT).as_posix()}")

print(pd.crosstab(pd.Series([c["language"] for c in candidates], name="language"),
                  pd.Series([c.get("tier", 1) for c in candidates], name="tier")))

In [ ]:
GEN_SYSTEM = """You write test questions for a retrieval experiment.
You get one passage in {language}. Pick ONE specific fact from it (a name, number, date, place, object
or the cause of something) and write:
- "question": one question in {language} asking for that fact
- "answer": the answer, 1 to 8 words, copied character for character from the passage
- "key_fact": one sentence in {language} that states the fact in the passage's own words and contains the answer

Rules for the question:
1. It must make sense on its own, without the passage: name the person, place, object or event it is about.
   Never refer to the passage itself ("the passage", "the text", "the article", "the author", "mentioned",
   "described"; in German "im Text", "der Artikel", "der Autor", "erwähnt", "genannt"), never use
   "this", "these", "he", "she", "it", "they" ("dieser", "diese", "er", "sie", "ihr") for something the
   question does not name, and never use "I", "we", "our" ("ich", "wir", "unser").
2. Reuse the passage's own words and phrases. Do not paraphrase or use synonyms.
3. Ask about a detail that only this passage contains, not general knowledge.
4. Do not put the answer in the question.
5. At most 30 words, exactly one question mark, at the end.
6. Do not ask about figure numbers, page numbers or literature references.

Reply only with JSON: {{"question": "...", "answer": "...", "key_fact": "..."}}"""

GEN_USER = 'Passage:\n"""\n{text}\n"""'

RETRY = """Your reply did not pass these checks:
{problems}
Write a new reply for the same passage that follows all rules. Reply only with JSON."""

PAR_SYSTEM = """You translate test questions from {source} to {target}.
You get a JSON object with "question", "answer" and "key_fact" in {source}. Translate each field into {target}.

Rules:
1. Translate faithfully and literally; the question stays a question.
2. Keep all names, numbers and dates exactly (only change the number format where {target} requires it).
3. The translated "answer" must appear inside the translated "key_fact".
4. Do not put the answer into the translated question.

Reply only with JSON: {{"question": "...", "answer": "...", "key_fact": "..."}}"""

RETRY_TRANSLATION = """Your translation did not pass these checks:
{problems}
Translate again, following all rules. Reply only with JSON."""

In [ ]:
SCHEMA = {
    "type": "object",
    "properties": {k: {"type": "string"} for k in ("question", "answer", "key_fact")},
    "required": ["question", "answer", "key_fact"],
}

def ask_model(messages, temperature, seed):
    """Send the conversation to Ollama; return the raw reply and the parsed fields."""
    response = requests.post(f"{OLLAMA}/api/chat", timeout=600, json={
        "model": MODEL,
        "messages": messages,
        "stream": False,
        "format": SCHEMA,   # Ollama forces the reply into this JSON shape
        "options": {"temperature": temperature, "seed": seed},
    })
    response.raise_for_status()
    raw = response.json()["message"]["content"]
    reply = json.loads(raw)
    return raw, {k: str(reply.get(k, "")).strip() for k in ("question", "answer", "key_fact")}

In [ ]:
def answer_pattern(answer):
    """Regex for the answer's words in order, ignoring case and punctuation/quotes between them."""
    words = re.findall(r"\w+", answer)
    if not words:
        return None
    return re.compile(r"\b" + r"\W+".join(map(re.escape, words)) + r"\b", re.IGNORECASE)

def contains(text, answer):
    pattern = answer_pattern(answer)
    return bool(pattern and pattern.search(text))

def find_span(answer, text):
    """The exact text of the chunk that matches the answer, or None."""
    pattern = answer_pattern(answer)
    match = pattern.search(text) if pattern else None
    return match.group(0) if match else None

def contains_loosely(text, answer):
    """True if most of the answer's words occur in the text, allowing different word endings
    (German inflection: "komplexe Behandlungen" ~ "komplexen Behandlungen")."""
    answer_words = [w.lower() for w in re.findall(r"\w+", answer)]
    text_stems = {w.lower()[:5] for w in re.findall(r"\w+", text)}
    if not answer_words:
        return False
    found = sum(w[:5] in text_stems for w in answer_words)
    return found / len(answer_words) >= MIN_ANSWER_MATCH

def word_overlap(text, chunk, lang):
    """Share of the text's content words that also occur in the chunk."""
    tokens = tokenize(text, lang)
    chunk_tokens = set(tokenize(chunk, lang))
    return sum(t in chunk_tokens for t in tokens) / max(len(tokens), 1)

def numbers(text):
    return sorted(re.findall(r"\d+", text))

FUNCTION_WORDS = {
    "DE": {"der", "die", "das", "und", "wer", "was", "wann", "wo", "welche", "welcher", "welches", "welchem",
           "welchen", "wie", "im", "von", "mit", "den", "dem", "des", "wurde", "wurden", "ist", "zu", "zur",
           "zum", "auf", "für", "ein", "eine", "einen", "einem", "einer", "gehört", "war", "waren"},
    "EN": {"the", "and", "who", "what", "when", "where", "which", "how", "of", "with", "was", "were", "is",
           "to", "did", "does", "for", "a", "an", "on", "belongs", "by"},
}

def is_language(text, lang):
    """Rough check: fails only if the other language's function words clearly dominate."""
    words = re.findall(r"\w+", text.lower())
    own = sum(w in FUNCTION_WORDS[lang] for w in words)
    other = sum(w in FUNCTION_WORDS[OTHER[lang]] for w in words)
    return own >= other

# Rule 1: words that only make sense if you can see the passage
BANNED = {
    "EN": r"\b(passage|this text|the text|article|excerpt|the author|according to|mentioned|described above"
          r"|this|these|those|he|she|it|its|his|her|him|they|them|their|we|us|our|my|me)\b",
    "DE": r"\b(Text|Textes|Abschnitt|Passage|Artikel|Artikels|Auszug|Autor|Autorin|laut|erwähnt\w*|genannt\w*"
          r"|beschrieben\w*|dies\w*|jene\w*|er|sie|ihm|ihn|ihr|ihre\w*|seine\w*|ich|wir|uns|unser\w*|mein\w*|mir|mich)\b",
}
# Rule 6: figure, page and literature references
REFERENCE = r"\b(Abb|Fig|Figure|Kat|Taf|Tafel|pl)\.?\s*\d+|\b(S|p|pp)\.\s*\d+"

def check_origin(item, chunk):
    """Rules 1-7 for the question/answer/key fact in the chunk's language."""
    lang, text = chunk["language"], chunk["text"]
    q, a, k = item["question"], item["answer"], item["key_fact"]
    problems, info = [], {}

    # Rule 6: form
    if not q.endswith("?") or q.count("?") != 1:
        problems.append("Write exactly one question, ending with a single '?'.")
    if len(q.split()) > MAX_QUESTION_WORDS:
        problems.append(f"The question is longer than {MAX_QUESTION_WORDS} words.")
    if re.search(REFERENCE, q):
        problems.append("Do not ask about figure, page or literature references.")

    # Rule 1: stands on its own
    hit = re.search(BANNED[lang], q, re.IGNORECASE)
    if hit:
        problems.append(f"The question must make sense without the passage; do not use '{hit.group(0)}', "
                        "name the person, place or object instead.")

    # Rules 2 and 3: answer copied from the chunk, not given away
    span = find_span(a, text)
    info["answer"] = span
    if span is None:
        problems.append("The answer is not copied exactly from the passage.")
    else:
        if len(span.split()) > MAX_ANSWER_WORDS:
            problems.append(f"The answer must be at most {MAX_ANSWER_WORDS} words.")
        if contains(q, span):
            problems.append("The question already contains the answer.")

    # Rule 4: wording close to the chunk
    overlap = word_overlap(q, text, lang)
    info["word_overlap"] = round(overlap, 2)
    if overlap < MIN_WORD_OVERLAP:
        problems.append(f"Only {overlap:.0%} of the question's content words come from the passage; "
                        "reuse the passage's own words.")

    # Rule 5: only this chunk answers it
    scores = index[lang]["bm25"].scores(tokenize(q, lang))
    ranked = [index[lang]["ids"][i] for i in sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)]
    info["gold_rank"] = ranked.index(chunk["chunk_id"]) + 1
    if info["gold_rank"] != 1:
        problems.append("The question also fits other passages; ask about a detail only this passage contains.")
    elif span and any(contains(chunk_text[cid], span) for cid in ranked[1:1 + UNIQUE_TOP_N]):
        problems.append("Other passages on the same topic contain the same answer; ask about a more specific detail.")

    # Rule 7: key fact
    if span and not contains(k, span):
        problems.append("The key_fact must contain the answer word for word.")
    k_overlap = word_overlap(k, text, lang)
    info["key_fact_overlap"] = round(k_overlap, 2)
    if k_overlap < MIN_WORD_OVERLAP:
        problems.append("The key_fact must use the passage's own words.")
    if len(k.split()) > MAX_KEY_FACT_WORDS:
        problems.append(f"The key_fact must be one sentence of at most {MAX_KEY_FACT_WORDS} words.")

    # Language
    if not is_language(q, lang) or not is_language(k, lang):
        problems.append(f"Write the question and key_fact in {LANG_NAME[lang]}.")
    return problems, info

def check_parallel(par, orig, target):
    """Rule 8 for the translated version."""
    q, a, k = par["question"], par["answer"], par["key_fact"]
    problems = []
    if not q.endswith("?") or q.count("?") != 1:
        problems.append("The translated question must end with a single '?'.")
    if not a:
        problems.append("The translated answer is empty.")
    elif not contains_loosely(k, a):
        problems.append("The translated answer must appear in the translated key_fact.")
    elif contains(q, a):
        problems.append("The translated question must not contain the answer.")
    for field in ("question", "answer", "key_fact"):
        if numbers(par[field]) != numbers(orig[field]):
            problems.append(f"Keep all numbers in '{field}' exactly as in the original.")
    if not is_language(q, target) or not is_language(k, target):
        problems.append(f"Translate into {LANG_NAME[target]}.")
    return problems

In [ ]:
EMPTY = {"question": "", "answer": "", "key_fact": ""}

def bullet_list(problems):
    return "\n".join(f"- {p}" for p in problems)

def seed_for(attempt):
    """Different seeds per run, so retried chunks get new answers from the model."""
    return SEED + 100 * (RUN - 1) + attempt

def generate(chunk):
    """Question, answer and key fact in the chunk's language; retries with feedback."""
    lang = chunk["language"]
    messages = [{"role": "system", "content": GEN_SYSTEM.format(language=LANG_NAME[lang])},
                {"role": "user", "content": GEN_USER.format(text=chunk["text"])}]
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            raw, item = ask_model(messages, TEMPERATURE, seed_for(attempt))
            problems, info = check_origin(item, chunk)
        except json.JSONDecodeError as error:
            raw, item, problems, info = "", dict(EMPTY), [f"The reply was not valid JSON ({error})."], {}
        if not problems:
            item["answer"] = info["answer"]  # store the exact text from the chunk
            return item, info, attempt, []
        messages += [{"role": "assistant", "content": raw},
                     {"role": "user", "content": RETRY.format(problems=bullet_list(problems))}]
    return item, info, attempt, problems

def translate(item, source, target):
    """Parallel version in the other language; retries with feedback."""
    messages = [{"role": "system", "content": PAR_SYSTEM.format(source=LANG_NAME[source], target=LANG_NAME[target])},
                {"role": "user", "content": json.dumps(item, ensure_ascii=False)}]
    for attempt in range(1, MAX_PARALLEL_ATTEMPTS + 1):
        try:
            raw, par = ask_model(messages, 0.0, seed_for(attempt))
            problems = check_parallel(par, item, target)
        except json.JSONDecodeError as error:
            raw, par, problems = "", dict(EMPTY), [f"The reply was not valid JSON ({error})."]
        if not problems:
            return par, attempt, []
        messages += [{"role": "assistant", "content": raw},
                     {"role": "user", "content": RETRY_TRANSLATION.format(problems=bullet_list(problems))}]
    return par, attempt, problems

## Run

Works through the candidates best first (tier 1 before tier 2), alternating languages, and stops per language at
`TARGET_PER_LANGUAGE`. Results are saved after every chunk: if you stop the cell, running it again continues where
it left off (keep `RUN` the same when you continue an interrupted run).
Expect roughly 10–40 seconds per chunk with Mistral 7B on a laptop.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Start a retry run: earlier rejections are moved aside (once per RUN) and tried again
if RETRY_REJECTED and REJECTED_PATH.exists():
    old = read_jsonl(REJECTED_PATH)
    if any(r.get("run", 1) < RUN for r in old):
        with open(PREVIOUS_REJECTED_PATH, "a", encoding="utf-8") as f:
            for r in old:
                r.setdefault("run", 1)
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        REJECTED_PATH.unlink()
        print(f"Moved {len(old)} earlier rejections to {PREVIOUS_REJECTED_PATH.name}; they are tried again.")

# Resume: skip chunks that already have a result
done, accepted = set(), collections.Counter()
if QUESTIONS_PATH.exists():
    for r in read_jsonl(QUESTIONS_PATH):
        done.add(r["chunk_id"])
        accepted[r["origin_lang"]] += 1
if REJECTED_PATH.exists():
    done.update(r["chunk_id"] for r in read_jsonl(REJECTED_PATH))

queue = sorted(candidates, key=lambda c: (c["pool_rank"], c["language"]))

for chunk in queue:
    lang, other = chunk["language"], OTHER[chunk["language"]]
    if chunk["chunk_id"] in done or accepted[lang] >= TARGET_PER_LANGUAGE:
        continue
    started = time.time()

    item, info, attempts, problems = generate(chunk)
    par, par_attempts, stage = {}, 0, "question"
    if not problems:
        par, par_attempts, problems = translate(item, lang, other)
        stage = "parallel"

    l, o = lang.lower(), other.lower()
    record = {
        "chunk_id": chunk["chunk_id"],
        "doc_id": chunk["doc_id"],
        "origin_lang": lang,
        "pool_rank": chunk["pool_rank"],
        "tier": chunk.get("tier", 1),
        f"question_{l}": item["question"], f"answer_{l}": item["answer"], f"key_fact_{l}": item["key_fact"],
        f"question_{o}": par.get("question", ""), f"answer_{o}": par.get("answer", ""),
        f"key_fact_{o}": par.get("key_fact", ""),
        "word_overlap": info.get("word_overlap"),
        "key_fact_overlap": info.get("key_fact_overlap"),
        "gold_rank": info.get("gold_rank"),
        "attempts": attempts,
        "parallel_attempts": par_attempts,
        "model": MODEL, "temperature": TEMPERATURE, "seed": SEED, "run": RUN,
    }
    if problems:
        record.update({"failed_at": stage, "problems": problems})
        path = REJECTED_PATH
    else:
        record["fact_id"] = f"f_{chunk['chunk_id']}"
        accepted[lang] += 1
        path = QUESTIONS_PATH
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

    status = "ok" if not problems else f"REJECTED at {stage}"
    print(f"{chunk['chunk_id']}: {status}, {attempts}+{par_attempts} attempts, "
          f"{time.time() - started:.0f}s | accepted DE {accepted['DE']}, EN {accepted['EN']}")

print("Accepted per language:", dict(accepted))
for lang in LANG_NAME:
    if accepted[lang] < TARGET_PER_LANGUAGE:
        print(f"Note: {lang} reached only {accepted[lang]} of {TARGET_PER_LANGUAGE} (candidate pool used up).")

In [ ]:
REVIEW_COLUMNS = ["approved", "notes", "auto_flags", "fact_id", "chunk_id", "origin_lang", "tier",
                  "question_de", "answer_de", "key_fact_de",
                  "question_en", "answer_en", "key_fact_en",
                  "word_overlap", "attempts", "chunk_text"]

def auto_flags(row):
    """Words that suggest a question does not stand on its own (also catches questions accepted
    before this rule existed). A flag is a hint for your review, not an automatic rejection."""
    flags = []
    for lang in ("DE", "EN"):
        hits = sorted({m.group(0).lower() for m in re.finditer(BANNED[lang], row[f"question_{lang.lower()}"], re.IGNORECASE)})
        if hits:
            flags.append(f"{lang}: " + ", ".join(hits))
    return "; ".join(flags)

questions = pd.DataFrame(read_jsonl(QUESTIONS_PATH))
questions["tier"] = questions.get("tier", pd.Series(1, index=questions.index)).fillna(1).astype(int)
questions["chunk_text"] = questions["chunk_id"].map(chunk_text)
questions["auto_flags"] = questions.apply(auto_flags, axis=1)
questions["approved"], questions["notes"] = "", ""

# Re-running keeps the approvals and notes you already entered
if REVIEW_PATH.exists():
    previous = pd.read_csv(REVIEW_PATH, dtype=str, keep_default_na=False, encoding="utf-8-sig",
                           sep=None, engine="python")  # also reads Excel's ';' separated files
    previous = previous.set_index("fact_id")
    for col in ("approved", "notes"):
        questions[col] = questions["fact_id"].map(previous[col]).fillna("")

REVIEW_DIR.mkdir(exist_ok=True)
(questions.sort_values(["origin_lang", "pool_rank"])[REVIEW_COLUMNS]
    .to_csv(REVIEW_PATH, index=False, encoding="utf-8-sig"))
print(f"Wrote {len(questions)} rows to {REVIEW_PATH.relative_to(ROOT)} "
      f"({(questions['auto_flags'] != '').sum()} with auto_flags)")

In [ ]:
rejected = (pd.DataFrame(read_jsonl(REJECTED_PATH)) if REJECTED_PATH.exists()
            else pd.DataFrame(columns=["origin_lang", "failed_at", "problems"]))

summary = pd.DataFrame({
    "accepted": questions.groupby("origin_lang").size(),
    "rejected (this run)": rejected.groupby("origin_lang").size(),
    "flagged": questions[questions["auto_flags"] != ""].groupby("origin_lang").size(),
}).fillna(0).astype(int)
summary["mean_attempts"] = questions.groupby("origin_lang")["attempts"].mean().round(2)
summary["mean_word_overlap"] = questions.groupby("origin_lang")["word_overlap"].mean().round(2)
summary

In [ ]:
if len(rejected):
    display(rejected.explode("problems")
            .groupby(["origin_lang", "failed_at"])["problems"].value_counts()
            .rename("count").to_frame())
else:
    print("No rejected chunks.")

In [ ]:
pd.set_option("display.max_colwidth", 150)
questions.groupby("origin_lang").head(3)[["origin_lang", "question_de", "answer_de", "question_en", "answer_en"]]

## Review, then push

1. Open `review/03_questions.csv` in Excel. For each row, write **yes** or **no** in `approved` and add notes if needed.
   Check: does the question make sense on its own (look closely at rows with `auto_flags`)? Is the answer right?
   Is the parallel version a faithful translation?
2. Save as **CSV UTF-8** (a `;` separator is fine).
3. Push:
```
git add notebooks/05_generate_questions.ipynb data/questions review/03_questions.csv
git commit -m "Step 5: questions with key facts and parallel versions"
git push
```
Only rows marked **yes** are used in the next steps; at least 60 per language are needed.